# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">2주차 · 취향이 비슷한 사람끼리 묶는다</mark>

지난 시간에는 가장 많이 담긴 종목 10개를 300명 모두에게 똑같이 보여 주었습니다.

점수는 0.2163이 나왔고, 알고리즘을 하나도 쓰지 않은 것 치고는 나쁘지 않았습니다.

그런데 마지막에 문제를 하나 보았습니다.

원금을 지키는 것이 최우선인 투자자와 레버리지 상품까지 담는 투자자에게, 우리는 **같은 근거로 만든 같은 목록**을 내밀었습니다.

오늘은 이 문제를 풉니다.

그 사람이 지금까지 담아 온 것을 보고 취향을 짐작해서, 사람마다 다른 목록을 만듭니다.

그리고 추천을 채점하는 방법이 하나가 아니라는 것을 알아보고, 세 가지 지표를 직접 계산해 봅니다.

---

### 오늘의 구성

| 파트 | 종류 | 시간 | 하는 일 |
|---|---|---|---|
| 1 | 개념 | 10분 | 지난 시간의 한계를 다시 보고, 무엇이 더 필요한지 정리합니다 |
| 2 | 실습 | 10분 | 거래 기록에서 주력 섹터와 평균 위험도를 뽑아 사람을 묶습니다 |
| 3 | 개념 | 10분 | 채점하는 방법 세 가지를 배웁니다 |
| 4 | 실습 | 10분 | 세 지표를 직접 계산하고 1주차와 비교합니다 |
| — | 질문 | 10분 | 노트북을 덮고 이야기합니다 |

### 강사용 · 예상 소요 시간

- 전체 셀을 처음부터 끝까지 실행 — **약 12초**
- 가장 오래 걸리는 셀은 마지막 비교 셀로 약 6초입니다.
- 첫 셀의 `import` 가 3~5초 걸리므로 **수업 시작 전에 미리 한 번 실행**해 두시길 권합니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

이 노트북은 **아무것도 설치하지 않고** 브라우저에서 바로 실행할 수 있습니다.

데이터도 따로 내려받지 않습니다. 아래 준비 셀이 알아서 가져옵니다.

### Google Colab 으로 열기

1. 아래 주소를 눌러 주세요.
   - https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/02_rules_metrics.ipynb
2. 구글 계정으로 로그인합니다.
   - 이미 로그인돼 있으면 이 단계는 그냥 지나갑니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
   - `Warning: This notebook was not authored by Google` 이라는 문구가 나옵니다.
   - 강사가 만든 파일이니 **`Run anyway`(그래도 실행)** 를 누르시면 됩니다.
4. **아래 "실습 준비" 셀의 ▶ 버튼을 누릅니다.**
   - 실습에 필요한 자료와 한글 글꼴을 받아옵니다. 20초쯤 걸립니다.
   - "준비 끝" 이 나오면 성공입니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.
> 누르지 않으면 탭을 닫을 때 사라집니다.

---

### <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>


In [1]:
# 이 셀의 목적 — Colab 이면 실습 자료와 한글 글꼴을 받아온다
# ── 실습 준비 ─────────────────────────────────────────────
# Colab 에서 열었을 때만 자료를 내려받습니다.
# 내 컴퓨터에서 열었다면 아무 일도 하지 않고 넘어갑니다.
import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:          # Colab 이면 이 안이 실행된다
    print("Colab 입니다. 실습 자료를 받아옵니다 ...")

    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면
        subprocess.run(["git", "-C", "/content/recsys",   # 최신으로 갱신한다
                        "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git",
                        "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("  자료 받기 완료 —", os.getcwd())

    print("한글 글꼴을 설치합니다 ...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"],   # 나눔고딕 설치
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # 설치 로그는 숨김

    # 새로 깐 글꼴을 matplotlib 이 알아보도록 글꼴 목록을 다시 읽게 합니다.
    subprocess.run(["rm", "-rf", os.path.expanduser("~/.cache/matplotlib")])
    print("  글꼴 설치 완료")

    print()
    print("준비 끝. 아래 셀부터 차례로 실행하세요.")
else:                                                 # 내 컴퓨터일 때
    print("내 컴퓨터에서 실행 중입니다. 따로 받아올 것이 없습니다.")
    print("지금 폴더:", os.getcwd())


내 컴퓨터에서 실행 중입니다. 따로 받아올 것이 없습니다.
지금 폴더: /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks


## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1 · 개념 — 무엇이 더 필요한가</mark>

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.1 지난 시간이 남긴 문제  `[PPT]`</mark>

지난 시간 마지막에 두 사람의 추천 목록을 나란히 놓아 보았습니다.

한 사람은 원금을 지키는 것이 최우선인 투자자였고, 다른 한 사람은 손실을 감수하고 큰 수익을 노리는 투자자였습니다.

두 목록에 차이가 있긴 했지만, 그것은 **이미 담은 종목을 걸러낸 결과**였을 뿐입니다.

순위표는 하나뿐이었고 두 사람 모두 그 표를 위에서부터 받았습니다.

추천의 근거가 사람에 따라 달라진 것이 아닙니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.2 사람마다 다르게 하려면 무엇이 필요한가  `[PPT]`</mark>

사람마다 다른 추천을 하려면 **그 사람이 어떤 사람인지**를 알아야 합니다.

그런데 그걸 어떻게 알아낼까요.

1. **직접 물어보기** — 가입할 때 투자 성향 설문을 받습니다.
   - 실제 금융 회사가 하는 방식입니다.
   - 다만 설문에서 안정형이라고 답한 사람이 레버리지 상품을 사는 일은 아주 흔합니다.
   - 그리고 우리가 앞으로 쓸 데이터에는 그런 설문 결과가 아예 없는 경우가 많습니다.
2. **행동을 보기** — 그 사람이 지금까지 담아 온 것을 봅니다.
   - 반도체 종목만 담아 온 사람에게는 반도체에서 인기 있는 것을 주면 됩니다.
   - 말보다 행동이 정직합니다.
   - 오늘은 이 방법으로 갑니다.

`users.csv` 파일에는 사실 투자 성향이 적혀 있습니다.

하지만 오늘 그것을 **쓰지 않습니다.**

그 값은 이 연습용 데이터를 만들 때 심어 둔 정답지이고, 그것을 쓰면 답을 보고 문제를 푸는 셈이 되기 때문입니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">1.3 오늘 만들 것 — 규칙 기반 추천  `[PPT]`</mark>

오늘 만들 방식을 **규칙 기반 추천**이라고 부릅니다.

모델을 학습시키지 않고 사람이 정한 규칙만으로 만드는 방식입니다.

1. 사람마다 거래 기록을 보고 **주력 섹터**를 찾습니다.
2. 같은 기록에서 **평균 위험도**도 구합니다.
3. 두 값을 합쳐 무리 이름을 만듭니다. 예를 들면 `반도체|4` 같은 것입니다.
4. 무리마다 인기 순위표를 따로 만들고, 각자 자기 무리의 목록을 받습니다.

같은 무리에 속한 사람끼리는 여전히 같은 목록을 받습니다.

그래도 300명 전원이 같은 목록을 받던 지난주보다는 나아집니다.

## <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2 · 실습 — 사람을 무리로 묶는다</mark>

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.1 오늘 나오는 말  `[PPT]`</mark>

1. **섹터 (sector)** — 종목이 속한 산업 분야
   - 반도체 · 배당 · 채권 · 레버리지처럼 성격이 비슷한 종목을 묶은 이름입니다.
   - `items.csv` 의 `sector` 칸에 들어 있습니다.
2. **위험도 (risk_level)** — 그 종목이 얼마나 크게 오르내리는지를 1~5로 매긴 숫자
   - 1은 국채처럼 잘 안 움직이는 것, 5는 레버리지처럼 크게 흔들리는 것입니다.
   - `items.csv` 의 `risk_level` 칸에 들어 있습니다.
3. **세그먼트 (segment)** — 취향이 비슷하다고 판단한 사람들의 묶음
   - 실무에서 **사용자 세그먼테이션**이라고 부르는 그 작업입니다.
   - 아래 코드에서는 우리말로 `무리` 라고 적었습니다. 같은 말입니다.
   - 오늘은 `주력섹터|평균위험도` 형태의 문자열을 무리 이름으로 씁니다.
   - 같은 이름을 가진 사람들이 한 무리입니다.
4. **피처 (feature)** — 사람이나 상품을 설명하려고 데이터에서 뽑아낸 값
   - 오늘 만드는 **주력 섹터**와 **평균 위험도**가 바로 피처입니다.
   - 이 피처 두 개로 세그먼트를 나눕니다.
5. **매핑 (mapping)** — 어떤 값을 다른 값으로 바꿔 주는 것
   - 종목 번호 `I044` 를 넣으면 섹터 `반도체` 가 나오게 하는 일입니다.
   - 지난 시간에 만든 "번호 → 이름" 사전과 같은 원리입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.2 준비 — 지난 시간 것을 다시 불러온다</mark>

지난 시간에 한 일을 그대로 다시 합니다.

데이터를 읽고, 시간으로 나누고, 종목 번호를 이름으로 바꿔 주는 사전을 만듭니다.

지난 시간에 했던 것이므로 설명 없이 한 번에 실행합니다.

In [2]:
# 준비 — 필요한 도구를 불러옵니다.
import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구

sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 칸 안쪽 폴더도 찾게 한다

import pandas as pd                         # 표를 다루는 도구. 앞으로 pd 라고 부른다
import numpy as np                          # 숫자 계산 도구. 앞으로 np 라고 부른다
import matplotlib                           # 그래프 도구 (설정용)
import matplotlib.pyplot as plt             # 그래프를 그리는 부분. plt 라고 부른다

import recsys                               # 이 수업용으로 만든 도구 모음

# 그래프의 한글이 깨지지 않도록, 이 컴퓨터에 실제로 깔려 있는 글꼴 중에서 고릅니다.
from matplotlib import font_manager         # 깔려 있는 글꼴 목록을 알려 주는 도구

설치된_글꼴 = {f.name for f in font_manager.fontManager.ttflist}   # 글꼴 이름만 모은다
후보 = ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]   # 찾아볼 순서
고른_글꼴 = next((이름 for 이름 in 후보 if 이름 in 설치된_글꼴), None)   # 후보 중 처음 발견된 것. 없으면 None

if 고른_글꼴:                                            # 하나라도 찾았으면
    matplotlib.rcParams["font.family"] = 고른_글꼴        # 그래프 글꼴로 지정한다
    matplotlib.rcParams["axes.unicode_minus"] = False    # 마이너스 기호가 네모로 나오는 것을 막는다
    print(f"한글 글꼴: {고른_글꼴}")
else:                                                    # 하나도 못 찾았으면
    print("한글 글꼴을 못 찾았습니다. 그래프의 한글이 네모로 보일 수 있습니다.")
    print("맥은 기본 설치돼 있고, 윈도우는 '맑은 고딕'이 있으면 됩니다.")

print("준비 완료")


한글 글꼴: AppleGothic
준비 완료


In [3]:
# 데이터를 읽고 시간으로 나눕니다. 지난 시간과 똑같습니다.
items, users, inter = recsys.load()                  # 종목·투자자·거래 기록 세 표를 한 번에 읽는다
train, test, cut = recsys.split_by_time(inter)       # 앞 11개월(train) / 마지막 1개월(test) / 자른 날짜(cut)

# 종목 번호를 이름으로 바꿔 주는 사전 (지난 시간에 만든 것)
종목표_번호색인 = items.set_index("item_id")          # 종목 번호를 행 이름으로 세운다
name_of = 종목표_번호색인["name"].to_dict()           # {번호: 이름} 사전으로 바꾼다

# 사람마다 학습 구간에서 이미 담은 종목들
사람별_담은종목 = train.groupby("user_id")["item_id"].apply(set)   # 사람별로 묶어 종목을 집합으로
seen_map = 사람별_담은종목.to_dict()                  # {사람: 담은 종목 집합} 사전

# 지난 시간의 추천 방식 — 누가 오든 인기 순위표 위에서부터 10개
종목별_거래건수 = train["item_id"].value_counts()     # 종목마다 기록이 몇 줄인지 센다(많은 순)
인기순위 = list(종목별_거래건수.index)                 # 번호만 많은 순으로 뽑아 목록으로


def 모두에게_같은_추천(user_id, seen):
    """누구에게나 같은 인기 순위표에서 아직 안 담은 것 10개를 준다."""
    return recsys.take(인기순위, seen)                # seen(이미 담은 것)을 빼고 위에서 10개


지난시간_점수, _ = recsys.recall_at_k(모두에게_같은_추천, train, test)   # 그 방식의 Recall@10

print(f"학습 {len(train):,}건 · 채점 {len(test):,}건")
print(f"지난 시간 방식의 점수  Recall@10 = {지난시간_점수:.4f}")


학습 7,095건 · 채점 991건
지난 시간 방식의 점수  Recall@10 = 0.2163


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.3 거래 기록에 종목 정보를 붙인다</mark>

거래 기록에는 종목 번호만 있습니다.

섹터와 위험도는 종목 표에 있으니, 번호를 보고 가져다 붙여야 합니다.

먼저 "번호 → 섹터" 와 "번호 → 위험도" 사전을 각각 만듭니다.

지난 시간에 "번호 → 이름" 사전을 만든 것과 똑같은 방법입니다.

In [4]:
# 섹터 사전 — 종목 번호를 넣으면 섹터가 나옵니다.
섹터_열 = 종목표_번호색인["sector"]     # DataFrame 에서 열 하나만 → Series
섹터_of = 섹터_열.to_dict()             # Series → dict. {번호: 섹터}

# 위험도 사전 — 같은 방법으로 하나 더
위험도_열 = 종목표_번호색인["risk_level"]   # 위험도 열만 꺼낸다
위험도_of = 위험도_열.to_dict()             # {번호: 1~5 숫자}

print(f"섹터 사전에 종목 {len(섹터_of)}개")
print()
print("I044 →", name_of["I044"], "· 섹터", 섹터_of["I044"], "· 위험도", 위험도_of["I044"])
print("I079 →", name_of["I079"], "· 섹터", 섹터_of["I079"], "· 위험도", 위험도_of["I079"])


섹터 사전에 종목 100개

I044 → 삼성전자 · 섹터 반도체 · 위험도 3
I079 → KODEX 레버리지 · 섹터 레버리지 · 위험도 5


이제 이 사전으로 거래 기록에 섹터 열을 새로 붙입니다.

`.map()` 은 열의 값 하나하나를 사전에 넣어서 나온 결과로 바꿔 줍니다.

`I044` 가 들어가면 `반도체` 가 나오는 식입니다.

In [5]:
# 1단계 — 거래 기록에서 종목 번호 열을 꺼냅니다.
거래_종목번호 = train["item_id"]              # 8천 줄짜리 번호 목록

# 2단계 — 각 번호를 섹터 이름으로 바꿉니다.
거래_섹터 = 거래_종목번호.map(섹터_of)         # map = 사전을 대고 값을 갈아 끼운다

# 3단계 — 원래 표에 그 열을 붙인 새 표를 만듭니다.
거래_섹터포함 = train.assign(sector=거래_섹터)  # assign = 열을 더한 새 표를 돌려준다

print("바꾸기 전 :", list(거래_종목번호.head(3)))
print("바꾼 뒤   :", list(거래_섹터.head(3)))
print()
거래_섹터포함.head()


바꾸기 전 : ['I043', 'I032', 'I008']
바꾼 뒤   : ['지수', '금융', '채권']



,user_id,item_id,ts,event,sector
0,U0245,I043,2025-08-31 10:00:00,view,지수
1,U0055,I032,2025-09-03 12:00:00,like,금융
2,U0055,I008,2025-09-04 11:00:00,view,채권
3,U0198,I084,2025-09-09 15:00:00,view,레버리지
4,U0198,I090,2025-09-10 00:00:00,like,친환경


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.4 사람마다 주력 섹터를 찾는다</mark>

이제 사람별로 묶어서, 각자 가장 많이 담은 섹터를 하나 고릅니다.

`groupby("user_id")` 는 **같은 사람의 기록끼리 모아라**는 뜻입니다.

모은 다음 각 사람의 섹터 목록에서 가장 자주 나온 것을 뽑습니다.

In [6]:
def 가장_많은_값(값들):
    """여러 값 중 가장 자주 나온 것 하나를 돌려준다."""
    센_결과 = 값들.value_counts()   # 값별로 몇 번 나왔는지 세고 많은 순으로 정렬

    # ✏️ 빈칸 1 — 몇 번째를 가져와야 "가장 많은 것" 이 될까요?
    #    센_결과 는 이미 많은 순으로 정렬돼 있습니다. 이름은 .index 에 들어 있습니다.
    몇번째 = 0        # ← 이 숫자를 바꿔 보세요. 1 로 바꾸면 무엇이 나올까요?

    return 센_결과.index[몇번째]     # index = 값의 이름 쪽. [0] 이면 가장 많은 것


# 사람별로 묶어서 섹터 열에 위 함수를 적용합니다.
사람별_묶음 = 거래_섹터포함.groupby("user_id")   # 같은 사람의 줄끼리 모은다
사람별_섹터목록 = 사람별_묶음["sector"]          # 그 사람이 담은 섹터들만 본다
주력섹터_시리즈 = 사람별_섹터목록.agg(가장_많은_값)   # agg = 묶음마다 위 함수를 한 번씩

주력섹터 = 주력섹터_시리즈.to_dict()             # {사람: 주력 섹터} 사전

print(f"{len(주력섹터)}명의 주력 섹터를 구했습니다.")
print()
print("U0001 의 주력 섹터 :", 주력섹터["U0001"])
print("U0003 의 주력 섹터 :", 주력섹터["U0003"])


285명의 주력 섹터를 구했습니다.

U0001 의 주력 섹터 : 금융
U0003 의 주력 섹터 : 레버리지


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.5 평균 위험도를 구해 무리 이름을 만든다</mark>

섹터만으로는 구분이 거칩니다.

같은 지수 ETF를 담더라도 안전한 것만 담는 사람과 레버리지까지 담는 사람은 다릅니다.

그래서 그 사람이 담은 종목들의 **평균 위험도**를 하나 더 씁니다.

In [7]:
# 1단계 — 종목 번호를 위험도 숫자로 바꾼 열을 붙입니다.
거래_위험도 = train["item_id"].map(위험도_of)          # 번호 → 1~5 숫자로 갈아 끼운다
거래_위험도포함 = train.assign(risk=거래_위험도)        # risk 라는 열을 더한 새 표

# 2단계 — 사람별로 묶어 평균을 냅니다.
사람별_위험도 = 거래_위험도포함.groupby("user_id")["risk"]   # 사람별 위험도 묶음
평균위험도_실수 = 사람별_위험도.mean()                  # 묶음마다 평균. 2.75 같은 소수가 나온다

# 3단계 — 반올림해서 정수로 만듭니다. 2.75 나 3.2 를 모두 3 으로 모읍니다.
평균위험도_반올림 = 평균위험도_실수.round(0)            # 2.75 → 3.0
평균위험도_정수 = 평균위험도_반올림.astype(int)         # 3.0 → 3 (소수점 없애기)

평균위험도 = 평균위험도_정수.to_dict()                  # {사람: 1~5 정수} 사전

print(f"U0001 : {평균위험도_실수['U0001']:.2f}  →  {평균위험도['U0001']}")
print(f"U0003 : {평균위험도_실수['U0003']:.2f}  →  {평균위험도['U0003']}")


U0001 : 2.75  →  3
U0003 : 4.00  →  4


두 값을 이어 붙이면 무리 이름이 됩니다.

사람이 285명이니 `for` 로 한 명씩 돌면서 사전을 채웁니다.

In [8]:
무리 = {}                                # 빈 사전을 하나 만들어 두고 채운다

for 사람 in 주력섹터:                      # 285명을 한 명씩 돈다
    섹터 = 주력섹터[사람]                  # 그 사람의 주력 섹터
    위험도 = 평균위험도[사람]              # 그 사람의 평균 위험도
    무리[사람] = f"{섹터}|{위험도}"        # 둘을 막대(|)로 이어 이름을 만든다

무리_종류 = set(무리.values())             # 집합으로 만들면 중복이 사라져 종류만 남는다

print(f"투자자 {len(무리)}명이 {len(무리_종류)}개 세그먼트로 나뉘었습니다.")
print()
print("U0001 →", 무리["U0001"])
print("U0003 →", 무리["U0003"])


투자자 285명이 15개 세그먼트로 나뉘었습니다.

U0001 → 금융|3
U0003 → 레버리지|4


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.6 지난주에 못 본 것 — 두 사람이 같은 목록을 받았다</mark>

지난 시간 마지막에 보려다 시간이 모자랐던 대목입니다. **오늘 이야기의 출발점이라 먼저 봅니다.**

성향이 정반대인 두 사람을 골라, **1주차 방식**으로 추천하면 어떤 목록을 받는지 나란히 놓습니다.


In [9]:
# 성향이 정반대인 두 사람을 한 명씩 고릅니다.
채점대상 = set(test["user_id"])            # 채점 구간에 기록이 있는 사람만 고를 수 있다

보수적인_사람들 = users[users["persona"] == "안정형"]["user_id"]        # 안정형인 사람들의 아이디
안정형 = [사람 for 사람 in 보수적인_사람들 if 사람 in 채점대상][0]      # 그중 채점 대상인 첫 사람

공격적인_사람들 = users[users["persona"] == "공격형"]["user_id"]        # 공격형인 사람들
공격형 = [사람 for 사람 in 공격적인_사람들 if 사람 in 채점대상][0]      # 그중 첫 사람

# 1주차 방식(모두에게 같은 목록)으로 두 사람에게 추천해 봅니다.
지난주_비교 = pd.DataFrame({                # 두 목록을 나란히 놓은 표를 만든다
    f"안정형 {안정형}": [name_of[i] for i in 모두에게_같은_추천(안정형, seen_map[안정형])],   # 번호를 이름으로
    f"공격형 {공격형}": [name_of[i] for i in 모두에게_같은_추천(공격형, seen_map[공격형])],
})
지난주_비교.index = range(1, 11)            # 행 이름을 1~10 등수로 바꾼다

print("1주차 방식 — 성향이 정반대인 두 사람이 받은 목록\n")
지난주_비교


1주차 방식 — 성향이 정반대인 두 사람이 받은 목록



,안정형 U0001,공격형 U0003
1,Vanguard FTSE Developed Markets ETF,Vanguard FTSE Developed Markets ETF
2,TIGER 미국S&P500,Vanguard S&P 500 ETF
3,Taiwan Semiconductor,KODEX 미국나스닥100
4,Invesco QQQ Trust,TIGER 리츠부동산인프라
5,Vanguard S&P 500 ETF,Schwab US Dividend Equity ETF
6,ACE 미국S&P500,DB하이텍
7,KODEX 미국나스닥100,KODEX 국고채30년액티브
8,Schwab US Dividend Equity ETF,Salesforce
9,ProShares UltraPro QQQ,TIGER 은행고배당플러스TOP10
10,DB하이텍,KODEX 미국배당다우존스


위쪽이 거의 같습니다. 몇 자리가 다른 것은 두 사람을 다르게 봐서가 아니라 **이미 담은 종목을 뺐기 때문**입니다.

안정형은 원금을 지키려는 사람이고 공격형은 위험을 감수하는 사람입니다. 성향이 정반대인데 같은 답을 받았습니다.

**이것을 추천이라고 부를 수 있을까요.** 이 불만이 오늘 할 일입니다. 아래에서 세그먼트별 목록을 만들어 다시 봅니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.7 세그먼트별 추천을 만들고 점수를 낸다</mark>

먼저 무리마다 인기 순위표를 따로 만듭니다.

그다음 각자 자기 무리의 순위표에서 10개를 받게 합니다.


In [10]:
# 거래 기록에 무리 이름을 붙입니다.
거래_무리 = train["user_id"].map(무리)              # 사람 아이디 → 세그먼트 이름
거래_무리포함 = train.assign(group=거래_무리)        # group 열을 더한 새 표

# 무리별로 인기 순위표를 따로 만듭니다.
무리별_순위표 = {}                                  # {세그먼트: 종목 순위 목록}

for 무리이름, 그_무리의_거래 in 거래_무리포함.groupby("group"):   # 세그먼트마다 한 번씩
    그_무리_종목건수 = 그_무리의_거래["item_id"].value_counts()    # 그 안에서만 센다
    무리별_순위표[무리이름] = list(그_무리_종목건수.index)          # 많은 순 번호 목록

print(f"순위표 {len(무리별_순위표)}개를 만들었습니다.")
print()
print("레버리지|4 세그먼트의 상위 3개 :")
for 번호 in 무리별_순위표["레버리지|4"][:3]:        # 그 세그먼트 순위표에서 위 3개만
    print("   ", name_of[번호])


순위표 15개를 만들었습니다.

레버리지|4 세그먼트의 상위 3개 :
    Direxion Semiconductor Bull 3X
    TIGER 미국나스닥100레버리지
    KODEX 코스닥150레버리지


In [11]:
def 무리별_추천(user_id, seen):
    """그 사람이 속한 세그먼트의 순위표에서 아직 안 담은 것 10개를 준다."""
    그_사람의_무리 = 무리.get(user_id)                       # 세그먼트 이름을 찾는다
    순위표 = 무리별_순위표.get(그_사람의_무리, 인기순위)      # 무리를 모르면 전체 인기순으로
    return recsys.take(순위표, seen)                         # 이미 담은 것을 빼고 10개


이번시간_점수, n_eval = recsys.recall_at_k(무리별_추천, train, test)   # 이번 방식의 Recall@10

오른_비율 = (이번시간_점수 - 지난시간_점수) / 지난시간_점수   # (새 점수 - 옛 점수) ÷ 옛 점수

print(f"지난 시간 (모두에게 같은 목록)  {지난시간_점수:.4f}")
print(f"이번 시간 (세그먼트별 목록)      {이번시간_점수:.4f}")
print()
print(f"{오른_비율:+.1%} 올랐습니다.   (채점 대상 {n_eval}명)")


지난 시간 (모두에게 같은 목록)  0.2163
이번 시간 (세그먼트별 목록)      0.2684

+24.1% 올랐습니다.   (채점 대상 266명)


지난 시간에 비교했던 두 사람을 다시 불러옵니다.

이번에는 두 사람이 **서로 다른 순위표**를 받으므로 근거 자체가 다릅니다.

In [12]:
# 아까 그 두 사람입니다 (위에서 이미 골라 뒀습니다).
print(f"{안정형} 이 속한 세그먼트 : {무리[안정형]}")
print(f"{공격형} 이 속한 세그먼트 : {무리[공격형]}")
print()

비교 = pd.DataFrame({                       # 이번에는 세그먼트별 추천으로 두 목록을 만든다
    f"{안정형} ({무리[안정형]})": [name_of[i] for i in 무리별_추천(안정형, seen_map[안정형])],
    f"{공격형} ({무리[공격형]})": [name_of[i] for i in 무리별_추천(공격형, seen_map[공격형])],
})
비교.index = range(1, 11)                    # 행 이름을 1~10 등수로

비교


U0001 이 속한 세그먼트 : 금융|3
U0003 이 속한 세그먼트 : 레버리지|4



,U0001 (금융|3),U0003 (레버리지|4)
1,TIGER 미국배당다우존스,TIGER 미국나스닥100레버리지
2,iShares 20+ Year Treasury Bond ETF,ProShares UltraPro Short QQQ
3,TIGER 은행고배당플러스TOP10,iShares Global Clean Energy ETF
4,신한지주,알테오젠
5,ACE 미국배당다우존스,LG에너지솔루션
6,ProShares UltraPro QQQ,Advanced Micro Devices
7,Vanguard Total Bond Market ETF,TIGER 리츠부동산인프라
8,Vanguard High Dividend Yield ETF,Vanguard FTSE Developed Markets ETF
9,KODEX 미국배당다우존스,KODEX 국고채30년액티브
10,Vanguard FTSE Developed Markets ETF,TIGER 2차전지테마


지난주와 달리 목록이 성격까지 갈렸습니다.

한쪽은 배당과 채권이, 다른 쪽은 레버리지와 테마 상품이 위로 올라옵니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">2.8 위험도를 빼면 점수가 어떻게 될까  ✏️ 직접 해 보기</mark>

지금 세그먼트 이름은 **주력 섹터 + 평균 위험도** 두 개로 만들었습니다.

그런데 위험도가 정말 필요했을까요? 섹터만으로 나눠도 되지 않을까요?

**직접 해 보고 점수로 확인합니다.** 아래 셀을 실행하면 두 방식의 점수가 나란히 나옵니다.


In [13]:
# ✏️ 빈칸 2 — 섹터만으로 나눈 세그먼트와 점수를 비교해 봅니다.

def 점수_내보기(무리사전):
    """무리 사전을 받아 세그먼트별 순위표를 만들고 Recall@10 을 돌려준다."""
    거래 = train.assign(group=train["user_id"].map(무리사전))   # 받은 사전으로 group 열을 붙인다

    표 = {}                                                     # {세그먼트: 순위 목록}
    for 이름, 그_거래 in 거래.groupby("group"):                  # 세그먼트마다
        표[이름] = list(그_거래["item_id"].value_counts().index)  # 그 안에서만 세어 순위를 만든다

    def 추천(user_id, seen):
        내_표 = 표.get(무리사전.get(user_id), 인기순위)   # 없으면 전체 인기순
        return recsys.take(내_표, seen)                  # 이미 담은 것을 빼고 10개

    점수, _ = recsys.recall_at_k(추천, train, test)      # 그 방식으로 채점
    return 점수


# 섹터만으로 만든 무리 사전 — 위험도를 붙이지 않는다
무리_섹터만 = {}                          # 빈 사전
for 사람 in 주력섹터:                      # 285명을 돌면서
    무리_섹터만[사람] = 주력섹터[사람]     # 섹터 이름만 그대로 넣는다 (|위험도 없음)

섹터_위험도_점수 = 점수_내보기(무리)        # 원래 방식 (섹터 + 위험도)
섹터만_점수 = 점수_내보기(무리_섹터만)      # 위험도를 뺀 방식

print(f"섹터 + 위험도   Recall@10 = {섹터_위험도_점수:.4f}   세그먼트 {len(set(무리.values()))}개")
print(f"섹터만         Recall@10 = {섹터만_점수:.4f}   세그먼트 {len(set(무리_섹터만.values()))}개")
print()
print("→ 어느 쪽이 높나요? 세그먼트 개수와 점수는 어떤 관계일까요?")


섹터 + 위험도   Recall@10 = 0.2684   세그먼트 15개
섹터만         Recall@10 = 0.2562   세그먼트 9개

→ 어느 쪽이 높나요? 세그먼트 개수와 점수는 어떤 관계일까요?


**생각해 볼 것** — 세그먼트를 잘게 쪼개면 점수가 계속 오를까요?

한 사람마다 세그먼트를 하나씩 만들면 어떻게 될지 떠올려 보세요.

그 사람의 기록만으로 순위표를 만들게 되므로 **아직 안 담은 것을 추천할 수가 없습니다.**

너무 굵게 나누면 남과 같은 목록을 받고, 너무 잘게 나누면 추천할 것이 없습니다. 그 사이를 찾는 것이 이 일의 어려운 점입니다.


## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">3 · 개념 — 채점하는 방법 세 가지</mark>


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 세 지표를 한 번에 계산하는 함수</mark>

한 사람씩 돌면서 세 지표를 구하고, 마지막에 평균을 냅니다.

앞에서 배운 집합 연산이 여기서 쓰입니다.

In [14]:
K = 10   # 추천 목록의 길이. Precision@10 · Recall@10 의 그 10 입니다.

# NDCG 는 자리마다 점수를 깎습니다. 그 깎는 값이 할인 계수입니다.
# 1등 1.000 · 5등 0.387 · 10등 0.289 — 아래로 갈수록 완만하게 줄어듭니다.
할인 = 1 / np.log2(np.arange(2, K + 2))   # arange(2,12) = 2~11. 로그를 취해 1을 나눈다


def 세_지표(추천함수, train, test, k=K):
    """Precision@k · Recall@k · NDCG@k 를 한 번에 계산한다."""

    사람별_담은것 = train.groupby("user_id")["item_id"].apply(set).to_dict()   # {사람: 이미 담은 집합}

    precision_모음 = []       # 사람마다의 Precision 을 모아 둘 목록
    recall_모음 = []          # Recall 쪽
    ndcg_모음 = []            # NDCG 쪽

    for 사람, 그_사람의_채점기록 in test.groupby("user_id"):   # 채점 구간의 사람을 한 명씩

        이미_담은것 = 사람별_담은것.get(사람, set())            # 학습 구간에 담은 것
        채점구간_담은것 = set(그_사람의_채점기록["item_id"])    # 마지막 한 달에 담은 것

        # 이미 담은 것은 정답에서 뺍니다
        정답 = 채점구간_담은것 - 이미_담은것                    # 빼기(-) = 차집합

        # 맞힐 것이 없는 사람은 채점에서 제외합니다.
        if len(정답) == 0:
            continue                                          # 다음 사람으로 넘어간다

        추천 = 추천함수(사람, 이미_담은것)[:k]                  # 이 사람에게 준 10개
        맞힌개수 = len(정답 & set(추천))          # 양쪽에 다 있는 것의 개수 (&는 교집합)

        # ✏️ 빈칸 3 — 분자는 둘 다 맞힌개수 로 같습니다. 분모만 다릅니다.
        #    k 는 우리가 준 개수(10), len(정답) 은 그 사람이 담은 개수입니다.
        #    두 줄을 서로 바꿔 넣으면 점수가 어떻게 되는지도 확인해 보세요.
        precision_분모 = k
        recall_분모 = len(정답)

        precision_모음.append(맞힌개수 / precision_분모)   # 이 사람의 Precision 을 담는다
        recall_모음.append(맞힌개수 / recall_분모)          # 이 사람의 Recall

        # NDCG — 맞힌 자리의 할인 계수를 더한 뒤, 가능한 최댓값으로 나눕니다.
        # (이 대목은 읽고 넘어가시면 됩니다. 자리 가중치를 곱하는 것뿐입니다.)
        얻은점수 = 0.0                            # 맞힌 자리의 가중치를 더해 갈 그릇
        자리 = 0                                  # 지금 몇 번째 자리를 보는지
        for 종목 in 추천:                          # 준 10개를 위에서부터
            if 종목 in 정답:                       # 그 자리에서 맞혔으면
                얻은점수 = 얻은점수 + 할인[자리]    # 그 자리의 가중치를 더한다
            자리 = 자리 + 1                        # 다음 자리로

        최대점수 = 할인[:min(len(정답), k)].sum()   # 위에서부터 다 맞혔을 때의 점수
        ndcg_모음.append(얻은점수 / 최대점수)       # 받은 점수 ÷ 받을 수 있던 최대

    return (float(np.mean(precision_모음)),        # 사람마다의 점수를 평균 낸다
            float(np.mean(recall_모음)),
            float(np.mean(ndcg_모음)))


print("함수를 만들었습니다.")


함수를 만들었습니다.


직접 짠 Recall 이 도구 상자의 값과 같은지 맞춰 봅니다.

값이 다르다면 대개 **이미 담은 종목을 정답에서 빼지 않은 것**이 원인입니다.

In [15]:
prec, rec, ndcg = 세_지표(무리별_추천, train, test)   # 셋을 한 번에 받는다

print(f"Precision@10 = {prec:.4f}")
print(f"Recall@10    = {rec:.4f}")
print(f"NDCG@10      = {ndcg:.4f}")
print()

# 직접 짠 Recall 이 recsys 모듈의 값과 같은지 맞춰 봅니다.
if abs(rec - 이번시간_점수) < 1e-9:                # 아주 작은 차이는 같은 것으로 본다
    print(f"직접 계산한 Recall {rec:.4f} 이 recsys 모듈 값 {이번시간_점수:.4f} 과 같습니다.")
else:
    print(f"값이 다릅니다. 직접 계산 {rec:.6f} · recsys 모듈 {이번시간_점수:.6f}")


Precision@10 = 0.0951
Recall@10    = 0.2684
NDCG@10      = 0.1923

직접 계산한 Recall 0.2684 이 recsys 모듈 값 0.2684 과 같습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.3 1주차와 2주차를 세 지표로 비교한다</mark>

In [16]:
지난시간_세지표 = 세_지표(모두에게_같은_추천, train, test)   # 1주차 방식을 세 지표로
이번시간_세지표 = 세_지표(무리별_추천, train, test)         # 2주차 방식을 세 지표로

결과 = pd.DataFrame(                                        # 두 줄짜리 표로 만든다
    [지난시간_세지표, 이번시간_세지표],                       # 값 (튜플 두 개)
    columns=["Precision@10", "Recall@10", "NDCG@10"],       # 칸 이름
    index=["1주차 · 모두에게 같은 목록", "2주차 · 세그먼트별 목록"],   # 행 이름
).round(4)                                                  # 소수 네 자리까지

결과


,Precision@10,Recall@10,NDCG@10
1주차 · 모두에게 같은 목록,0.0793,0.2163,0.1711
2주차 · 세그먼트별 목록,0.0951,0.2684,0.1923


세 지표 모두에서 2주차가 나았습니다.

지표를 바꿔도 결론이 뒤집히지 않았다는 뜻이므로, 이번 개선은 믿을 만합니다.

다만 항상 이렇지는 않습니다.

어떤 모델은 Precision 은 오르는데 Recall 은 떨어지기도 합니다.

그래서 **무엇을 중요하게 여길지 미리 정해 두는 것**이 중요합니다.

In [17]:
# 이번 주 점수를 점수판 파일에 남깁니다. 10주 내내 이 기록과 견줍니다.
recsys.record(2, "취향이 비슷한 세그먼트끼리", 이번시간_점수,   # 주차 번호 · 방식 이름 · 점수
              note="거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다")


레벨 2 · 취향이 비슷한 세그먼트끼리 · Recall@10 = 0.2684


{'level': 2,
 'name': '취향이 비슷한 세그먼트끼리',
 'recall_at_10': 0.2684,
 'note': '거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다'}

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.4 오늘의 마무리  `[PPT]`</mark>

거래 기록만 보고 사람을 무리로 나누었더니 점수가 올랐습니다.

성향 설문 같은 것을 따로 받지 않고도 "이 사람이 무엇을 좋아하는지"를 어느 정도 짐작할 수 있다는 뜻입니다.

그런데 아직 한계가 남아 있습니다.

**같은 무리에 속한 사람들은 여전히 똑같은 목록을 받습니다.**

`지수|3` 세그먼트에 89명이 있다면 그 89명은 완전히 같은 추천을 받습니다.

사람마다 다른 추천이 되긴 했지만 무리 단위로만 달라진 것입니다.

그리고 더 신경 쓰이는 것이 하나 있습니다.

우리가 지금까지 재 온 점수가 **정말 믿을 만한 숫자인지** 아직 확인해 보지 않았습니다.

다음 시간에는 모델을 하나도 바꾸지 않고 채점 방법만 다시 들여다봅니다.

그런데 점수가 떨어집니다.
